In [ ]:
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)
import sys
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset as ncDataset
from datetime import datetime, timedelta

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"
archivos = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
with ncDataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

In [ ]:
# ==========================================
# DATASET TEMPORAL (9 canales): [vars en t | t-(t-1) | (t-1)-(t-2)]
# Los nombres salen del stats.npz generado en Datasets_Temporal.ipynb (Fase 5)
# ==========================================
DIR_DATASET     = '../dataset_temporal'      # el dataset original (3 canales) queda en '../dataset'
NOMBRES_CANALES = [str(n) for n in np.load(f'{DIR_DATASET}/stats.npz')['canales']]
N_CANALES       = len(NOMBRES_CANALES)       # 9
print(N_CANALES, NOMBRES_CANALES)

In [ ]:
# =====================================================================
# DATASETS
# =====================================================================

ruta_train = f'{DIR_DATASET}/train'
ruta_val   = f'{DIR_DATASET}/val'
ruta_stats = f'{DIR_DATASET}/stats.npz'

print("Mapeando archivos y cargando estadísticas globales...")
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset   = TormentaDataset(split_dir=ruta_val,   stats_path=ruta_stats)
print(f"-> Train: {len(train_dataset)} muestras | Val: {len(val_dataset)} muestras")
print(f"-> min_y: {train_dataset.min_y:.4f} | max_y: {train_dataset.max_y:.4f}")

vacias = 0
for _, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
    vacias += (y.flatten(1).amax(dim=1) < 1e-6).sum().item()
print(f"-> Targets vacíos en train: {vacias} / {len(train_dataset)} ({100*vacias/len(train_dataset):.2f}%)")

In [ ]:
class BobModel(nn.Module):
    """
    Modelo de referencia que ignora completamente la entrada
    y siempre predice LIGHTDENS = 0 en todos los puntos de grilla.
    """
    def __init__(self):
        super(BobModel, self).__init__()
        self.register_buffer('valor_constante', torch.log(torch.tensor(0.1)))

    def forward(self, x):
        batch_size, _, H, W = x.shape
        salida = self.valor_constante.expand(batch_size, 1, H, W)
        return salida


modelo_bob = BobModel().to(device)
print(modelo_bob)

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, in_channels=6):
        super(BaselineCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        
        self.salida = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        out = self.salida(x)
        return out

modelo_baseline = BaselineCNN(in_channels=6).to(device)
print(modelo_baseline)

In [ ]:
class UNet_WRF(nn.Module):
    """
    U-Net con profundidad y ancho configurables.

    Parámetros
    ----------
    in_channels : int, default 6
        Cantidad de canales de entrada (variables WRF).
    n_niveles : int, default 2
        Cantidad de niveles de encoder/decoder (pooling + upsampling).
        La arquitectura original tenía 2 niveles.
    n_filtros_base : int, default 32
        Cantidad de filtros en el primer nivel del encoder. Cada nivel
        siguiente duplica esta cantidad (32→64→128→...), igual que el
        bottleneck, que duplica una vez más el último nivel.
    kernel_sizes : list[int] o None
        Un kernel_size por nivel del encoder (y su espejo en el decoder).
        Si es None, usa kernel_size=3 en todos los niveles.
        Debe tener longitud == n_niveles.
    dropout_ps : list[float] o None
        Un dropout_p por nivel del encoder (y su espejo en el decoder),
        aplicado también en el bottleneck (usa el último valor de la lista).
        Si es None, no aplica dropout en ningún nivel (comportamiento original).
        Debe tener longitud == n_niveles.
    input_size : int, default 297
        Tamaño espacial (H=W) de la imagen de entrada, usado para calcular
        el padding necesario según la profundidad elegida.
    """
    def __init__(self, in_channels=6, n_niveles=2, n_filtros_base=32,
                 kernel_sizes=None, dropout_ps=None, input_size=297):
        super(UNet_WRF, self).__init__()

        if kernel_sizes is None:
            kernel_sizes = [3] * n_niveles
        assert len(kernel_sizes) == n_niveles, \
            f"kernel_sizes debe tener {n_niveles} elementos, tiene {len(kernel_sizes)}"

        if dropout_ps is None:
            dropout_ps = [0.0] * n_niveles
        assert len(dropout_ps) == n_niveles, \
            f"dropout_ps debe tener {n_niveles} elementos, tiene {len(dropout_ps)}"
        
        self.n_niveles = n_niveles
        self.dropout_ps = dropout_ps

        # --- Padding espacial dinámico según profundidad ---
        divisor = 2 ** n_niveles
        size_ajustado = ((input_size // divisor) + 1) * divisor
        pad_total = size_ajustado - input_size
        # Repartimos el padding: menos a la izquierda/arriba, más a la derecha/abajo
        self.pad_izq_arr = pad_total // 2
        self.pad_der_aba = pad_total - self.pad_izq_arr
        self.input_size = input_size

        # --- Canales por nivel
        canales = [n_filtros_base * (2 ** i) for i in range(n_niveles)]
        canal_bottleneck = n_filtros_base * (2 ** n_niveles)
        # Kernel del bottleneck: reusamos el del nivel más profundo
        k_bottleneck = kernel_sizes[-1]
        dropout_bottleneck = dropout_ps[-1]

        def bloque_conv(canal_in, canal_out, k, dropout_p):
            pad = k // 2
            capas = [
                nn.Conv2d(canal_in, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
                nn.Conv2d(canal_out, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
            ]
            if dropout_p > 0:
                capas.append(nn.Dropout2d(p=dropout_p))
            return nn.Sequential(*capas)
        
        # --- ENCODER (dinámico) ---
        self.enc_blocks = nn.ModuleList()
        self.pools      = nn.ModuleList()
        canal_in = in_channels
        for canal_out, k, p in zip(canales, kernel_sizes, dropout_ps):
            self.enc_blocks.append(bloque_conv(canal_in, canal_out, k, p))
            self.pools.append(nn.MaxPool2d(2, 2))
            canal_in = canal_out

        # --- BOTTLENECK ---
        self.bottleneck = bloque_conv(canal_in, canal_bottleneck, k_bottleneck, dropout_bottleneck)

        # --- DECODER (dinámico, en orden inverso al encoder) ---
        self.ups        = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        canal_in = canal_bottleneck
        for canal_skip, k, p in zip(reversed(canales), reversed(kernel_sizes), reversed(dropout_ps)):
            self.ups.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
            self.dec_blocks.append(bloque_conv(canal_in + canal_skip, canal_skip, k, p))
            canal_in = canal_skip

        # --- SALIDA ---
        self.salida  = nn.Conv2d(canal_in, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (self.pad_izq_arr, self.pad_der_aba,
                           self.pad_izq_arr, self.pad_der_aba))

        # --- Encoder ---
        skips = []
        h = x_pad
        for bloque, pool in zip(self.enc_blocks, self.pools):
            h = bloque(h)
            skips.append(h)       # guardamos para el skip connection
            h = pool(h)

        # --- Bottleneck ---
        h = self.bottleneck(h)

        # --- Decoder ---
        for up, bloque, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = bloque(h)

        out = self.salida(h)
        out = self.sigmoid(out)  # SOLO para FACL

        # Recortar el padding agregado al principio
        out = out[:, :, self.pad_izq_arr:self.pad_izq_arr + self.input_size,
                         self.pad_izq_arr:self.pad_izq_arr + self.input_size]
        return out

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1

def graficar_comparacion_modelos(diccionario_modelos, dataset, idx, vmax_percentil=99, modelos_facl=None):
    if len(diccionario_modelos) != 3:
        raise ValueError(f"Se esperaban 3 modelos, pero se pasaron {len(diccionario_modelos)}.")

    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    X, y = dataset[idx]
    y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
    vmax_real = np.percentile(y_real[y_real > 0], vmax_percentil) if (y_real > 0).any() else 1

    predicciones = {}
    for nombre, modelo in diccionario_modelos.items():
        modelo.eval()
        with torch.no_grad():
            pred_out = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()
            es_facl = (nombre in modelos_facl) if modelos_facl is not None else ('facl' in nombre.lower())
            if es_facl:
                pred_real = dataset.desnormalizar_y(pred_out).numpy()
            else:
                pred_real = torch.clamp(torch.exp(pred_out) - 0.1, min=0).numpy()
            predicciones[nombre] = pred_real

    # [CAMBIO] Relajamos un poco el ancho a 16 (antes estaba en 14 muy apretado, o 22 muy suelto)
    fig, axes = plt.subplots(1, 4, figsize=(16, 5), subplot_kw={'projection': CART_PROJ})
    axes = axes.flatten()

    lons_ticks = np.arange(int(LONS.min()), int(LONS.max()) + 1, 1)
    lats_ticks = np.arange(int(LATS.min()), int(LATS.max()) + 1, 1)
    extent = [LONS.min(), LONS.max(), LATS.min(), LATS.max()]

    gridliners = []

    def formatear_mapa(ax, es_panel_izquierdo, es_panel_inferior):
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')

        gl = ax.gridlines(crs=crs.PlateCarree(), draw_labels=True,
                          linewidth=0.3, color='gray', linestyle='--', alpha=0.3,
                          x_inline=False, y_inline=False)
        
        gl.xlocator = mticker.FixedLocator(lons_ticks)
        gl.ylocator = mticker.FixedLocator(lats_ticks)
        gl.xformatter = LongitudeFormatter(zero_direction_label=True)
        gl.yformatter = LatitudeFormatter()
        gl.xlabel_style = {'size': 11}
        gl.ylabel_style = {'size': 11}
        
        gl.top_labels = False
        gl.right_labels = False
        gl.bottom_labels = es_panel_inferior
        gl.left_labels = es_panel_izquierdo

        gridliners.append((gl, es_panel_izquierdo, es_panel_inferior))

    # --- Panel (a): Real ---
    ax_real = axes[0]
    formatear_mapa(ax_real, es_panel_izquierdo=True, es_panel_inferior=True)
    cf = ax_real.contourf(LONS, LATS, y_real, transform=crs.PlateCarree(),
                          cmap='hot_r', levels=np.linspace(0, vmax_real, 21),
                          vmin=0, vmax=vmax_real, extend='max')
    ax_real.set_title('(a) WRF-ELEC', fontsize=15, pad=10)

    # --- Paneles (b) a (d) ---
    letras = ['(b)', '(c)', '(d)']
    for i, (nombre, y_pred) in enumerate(predicciones.items()):
        pos = i + 1
        ax = axes[pos]
        
        es_panel_inferior = True
        es_panel_izquierdo = False

        formatear_mapa(ax, es_panel_izquierdo=es_panel_izquierdo, es_panel_inferior=es_panel_inferior)

        ax.contourf(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                    cmap='hot_r', levels=np.linspace(0, vmax_real, 21),
                    vmin=0, vmax=vmax_real, extend='max')
        ax.set_title(f'{letras[i]} {nombre}', fontsize=15, pad=10)

    # [CAMBIO] Le damos un respiro mínimo con wspace=0.05
    fig.subplots_adjust(hspace=0.0, wspace=0.05)
    fig.canvas.draw()
    
    for gl, es_izq, es_inf in gridliners:
        for artist in gl.label_artists:
            texto = artist.get_text()
            es_longitud = ('W' in texto or 'E' in texto)
            if es_longitud and not es_inf:
                artist.set_visible(False)
            if not es_longitud and not es_izq:
                artist.set_visible(False)

    cbar = fig.colorbar(cf, ax=axes.tolist(), orientation='horizontal', 
                        shrink=0.5, pad=0.18, aspect=40)
    cbar.set_label('LIGHTDENS', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    ruta_guardado = generar_ruta_imagen(carpeta="../images", prefijo="comparacion_modelos", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Gráfico guardado exitosamente en: {ruta_guardado}")
    
    plt.show()

    print("-" * 50)
    print(f"Estadísticas del Timestep {idx}")
    print(f"Real        — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    for nombre, y_pred in predicciones.items():
        print(f"{nombre:<11} — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")
        
def adaptar_pesos_unet(state_dict_viejo):
    """Traduce las llaves de la U-Net vieja a la estructura de la U-Net nueva."""
    mapeo = {
        'enc_conv1_1': 'enc_blocks.0.0',
        'enc_conv1_2': 'enc_blocks.0.2',
        'enc_conv2_1': 'enc_blocks.1.0',
        'enc_conv2_2': 'enc_blocks.1.2',
        'bottleneck_1': 'bottleneck.0',
        'bottleneck_2': 'bottleneck.2',
        'dec_conv1_1': 'dec_blocks.0.0',
        'dec_conv1_2': 'dec_blocks.0.2',
        'dec_conv2_1': 'dec_blocks.1.0',
        'dec_conv2_2': 'dec_blocks.1.2',
    }
    
    state_dict_nuevo = {}
    for key, value in state_dict_viejo.items():
        nueva_key = key
        # Si el nombre viejo está en nuestro mapa, lo reemplazamos
        for viejo, nuevo in mapeo.items():
            if viejo in key:
                nueva_key = key.replace(viejo, nuevo)
                break
        state_dict_nuevo[nueva_key] = value
        
    return state_dict_nuevo

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep.

    Parámetros
    ----------
    y_real, y_pred : np.ndarray 2D

    Returns
    -------
    dict con: mse, rmse, ssim
    """
    error = y_real - y_pred
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())
    valor_ssim = ssim(y_real, y_pred, data_range=data_range) if data_range > 0 else 1.0
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim}

def evaluar_dataset(modelo, dataset, data_range=None, es_facl=False):
    """
    Recorre todo el dataset, calcula las métricas de verificación para cada
    timestep y las devuelve en un DataFrame (una fila por timestep).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_out = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        
        # --- LÓGICA DE DESNORMALIZACIÓN DIFERENCIADA ---
        if es_facl:
            y_pred = dataset.desnormalizar_y(pred_out).numpy()
        else:
            y_pred = torch.clamp(torch.exp(pred_out) - 0.1, min=0).numpy()

        metricas = calcular_metricas_timestep(y_real, y_pred, data_range=data_range)
        metricas['idx'] = idx
        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Modelo', ylabel=None, color='#722F37'):
    """
    Grafica la evolución de una métrica de verificación a lo largo de los
    timesteps del conjunto de validación, con la media como referencia.
    """
    serie = df_metricas[['idx', columna]].dropna(subset=[columna]) # <-- ignora NaN
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(serie['idx'], serie[columna], 'o-', color=color, markersize=4)

    media = serie[columna].mean()
    ax.axhline(media, color='gray', linestyle='--', alpha=0.6, label=f'Media: {media:.3f}')

    ax.set_xlabel('Timestep (val)')
    ax.set_ylabel(ylabel or columna.upper())
    ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de {columna.upper()} guardada en: {ruta_guardado}")

    plt.show()
    return media

def graficar_mse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE (flashes/columna)')

def graficar_mae(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM')

def calcular_r2_global(modelo, dataset, nombre_modelo='Modelo', ignorar_borde=0, es_facl=False):
    """
    Calcula R² como métrica global sobre todos los píxeles activos
    del dataset, para un único modelo.

    Parámetros
    ----------
    modelo         : modelo PyTorch ya cargado y en eval mode.
    dataset        : TormentaDataset (normalmente val_dataset).
    nombre_modelo  : str, usado para armar el nombre del archivo de log/salida.
    ignorar_borde  : int, default 0
        Cantidad de píxeles a recortar en cada borde de la imagen antes de
        calcular R². Sirve para chequear si el mal desempeño en los bordes
        está arrastrando el R² global hacia abajo.
    es_facl        : bool, default False
        Indica si el modelo usa la función de pérdida FACL (para aplicar la
        desnormalización correcta).

    Returns
    -------
    float, el valor de R² calculado.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc=f"Calculando R² — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_out = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()

        # --- LÓGICA DE DESNORMALIZACIÓN DIFERENCIADA ---
        if es_facl:
            y_pred = dataset.desnormalizar_y(pred_out).numpy()
        else:
            y_pred = torch.clamp(torch.exp(pred_out) - 0.1, min=0).numpy()

        # --- Recorte de bordes, si corresponde ---
        if ignorar_borde > 0:
            y_real = y_real[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]
            y_pred = y_pred[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]

        mascara = y_real > 0
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    etiqueta_borde = f" (ignorando {ignorar_borde}px de borde)" if ignorar_borde > 0 else " (imagen completa)"
    print(f"R² {nombre_modelo}{etiqueta_borde} — sobre {len(y_real_all):,} píxeles activos: {r2:.4f}")

    # --- Guardado en archivo, nombrado según nombre_modelo ---
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    sufijo_borde = f"_sinborde{ignorar_borde}" if ignorar_borde > 0 else "_completo"
    ruta_log = f"../modelos/r2_{nombre_limpio}{sufijo_borde}.txt"
    with open(ruta_log, 'w', encoding='utf-8') as f:
        f.write(f"Modelo: {nombre_modelo}\n")
        f.write(f"Recorte de borde: {ignorar_borde}px\n")
        f.write(f"Píxeles activos evaluados: {len(y_real_all):,}\n")
        f.write(f"R²: {r2:.6f}\n")
    print(f"Resultado guardado en: {ruta_log}")

    residuos = y_pred_all - y_real_all

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(residuos, bins=100, color='#722F37', edgecolor='none', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Error = 0')
    ax.axvline(residuos.mean(), color='gold', linestyle='-', linewidth=2,
               label=f'Sesgo medio: {residuos.mean():+.2f}')
    ax.set_xlabel('Residuo (Predicho − Real)')
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'Distribución de residuos — {nombre_modelo}{etiqueta_borde}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    ruta_hist = f"../images/residuos_{nombre_limpio}{sufijo_borde}.png"
    plt.savefig(ruta_hist, dpi=300, bbox_inches='tight')
    print(f"Histograma guardado en: {ruta_hist}")
    plt.show()

    return r2

def buscar_mejores_ejemplos(diccionario_modelos, dataset, modelos_facl=['U-Net Comb V2'], top_n=5, umbral_actividad=1.0):
    """
    Busca los índices del dataset donde el modelo 'U-Net Comb V2' supera por mayor 
    margen a los otros modelos en términos de Error Cuadrático Medio (MSE).
    """
    resultados = []
    modelo_estrella = 'U-Net Comb V2'
    
    # Identificar cuáles son los modelos con los que competimos
    modelos_base = [m for m in diccionario_modelos.keys() if m != modelo_estrella]
    
    print(f"Evaluando {len(dataset)} timesteps para encontrar el mejor ejemplo...")
    print(f"Comparando {modelo_estrella} vs {modelos_base}")

    for idx in range(len(dataset)):
        X, y = dataset[idx]
        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()

        # Filtro: Ignorar timesteps donde casi no hay actividad eléctrica real.
        # Ajustá 'umbral_actividad' si ves que descarta mucho o poco.
        if y_real.max() < umbral_actividad:
            continue

        errores = {}
        for nombre, modelo in diccionario_modelos.items():
            modelo.eval()
            with torch.no_grad():
                # Asumo que 'device' está definido globalmente en tu notebook
                pred_out = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()
                
                es_facl = (nombre in modelos_facl)
                if es_facl:
                    pred_real = dataset.desnormalizar_y(pred_out).numpy()
                else:
                    pred_real = torch.clamp(torch.exp(pred_out) - 0.1, min=0).numpy()

                # Calculamos el error cuadrático medio (MSE) de este modelo
                mse = np.mean((y_real - pred_real) ** 2)
                errores[nombre] = mse

        # Queremos que el error de Comb V2 sea el más bajo posible, 
        # y el promedio de los otros dos sea lo más alto posible.
        error_estrella = errores[modelo_estrella]
        error_promedio_base = np.mean([errores[m] for m in modelos_base])
        
        # La "puntuación" es la diferencia de errores. A mayor diferencia, mejor para V2
        ventaja = error_promedio_base - error_estrella

        resultados.append({
            'idx': idx,
            'ventaja': ventaja,
            'mse_estrella': error_estrella,
            'mse_base': error_promedio_base
        })

    # Ordenar de mayor ventaja a menor ventaja
    resultados_ordenados = sorted(resultados, key=lambda x: x['ventaja'], reverse=True)

    print("\n--- ¡Búsqueda finalizada! ---")
    print(f"Top {top_n} mejores índices para mostrar:")
    for i in range(min(top_n, len(resultados_ordenados))):
        res = resultados_ordenados[i]
        print(f"Puesto {i+1} -> Índice: {res['idx']} | Ventaja MSE: {res['ventaja']:.4f} "
              f"(MSE V2: {res['mse_estrella']:.4f} vs Promedio Otros: {res['mse_base']:.4f})")
        
    return resultados_ordenados

def graficar_reconstruccion_individual(modelo, nombre_modelo, dataset, idx, vmax_percentil=99, es_facl=True):
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    X, y = dataset[idx]
    # Extraemos la verdad de terreno (WRF-ELEC)
    y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
    vmax_real = np.percentile(y_real[y_real > 0], vmax_percentil) if (y_real > 0).any() else 1

    # Obtenemos la predicción del modelo
    modelo.eval()
    with torch.no_grad():
        pred_out = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()
        if es_facl:
            pred_real = dataset.desnormalizar_y(pred_out).numpy()
        else:
            pred_real = torch.clamp(torch.exp(pred_out) - 0.1, min=0).numpy()

    # [CAMBIO] Ancho reducido de 11.5 a 10 para compensar que ahora están más juntas
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={'projection': CART_PROJ})
    axes = axes.flatten()

    lons_ticks = np.arange(int(LONS.min()), int(LONS.max()) + 1, 1)
    lats_ticks = np.arange(int(LATS.min()), int(LATS.max()) + 1, 1)

    gridliners = []

    def formatear_mapa(ax, es_panel_izquierdo, es_panel_inferior):
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')

        gl = ax.gridlines(crs=crs.PlateCarree(), draw_labels=True,
                          linewidth=0.3, color='gray', linestyle='--', alpha=0.3,
                          x_inline=False, y_inline=False)
        
        gl.xlocator = mticker.FixedLocator(lons_ticks)
        gl.ylocator = mticker.FixedLocator(lats_ticks)
        gl.xformatter = LongitudeFormatter(zero_direction_label=True)
        gl.yformatter = LatitudeFormatter()
        gl.xlabel_style = {'size': 11}
        gl.ylabel_style = {'size': 11}
        
        gl.top_labels = False
        gl.right_labels = False
        gl.bottom_labels = es_panel_inferior
        gl.left_labels = es_panel_izquierdo

        gridliners.append((gl, es_panel_izquierdo, es_panel_inferior))

    # --- Panel (a): WRF-ELEC (Real) ---
    ax_real = axes[0]
    formatear_mapa(ax_real, es_panel_izquierdo=True, es_panel_inferior=True)
    cf = ax_real.contourf(LONS, LATS, y_real, transform=crs.PlateCarree(),
                          cmap='hot_r', levels=np.linspace(0, vmax_real, 21),
                          vmin=0, vmax=vmax_real, extend='max')
    ax_real.set_title('(a) WRF-ELEC', fontsize=15, pad=10)

    # --- Panel (b): U-Net Comb V2 ---
    ax_pred = axes[1]
    formatear_mapa(ax_pred, es_panel_izquierdo=False, es_panel_inferior=True)
    ax_pred.contourf(LONS, LATS, pred_real, transform=crs.PlateCarree(),
                    cmap='hot_r', levels=np.linspace(0, vmax_real, 21),
                    vmin=0, vmax=vmax_real, extend='max')
    ax_pred.set_title(f'(b) {nombre_modelo}', fontsize=15, pad=10)

    fig.subplots_adjust(hspace=0.0, wspace=0.2)
    fig.canvas.draw()
    
    for gl, es_izq, es_inf in gridliners:
        for artist in gl.label_artists:
            texto = artist.get_text()
            es_longitud = ('W' in texto or 'E' in texto)
            if es_longitud and not es_inf:
                artist.set_visible(False)
            if not es_longitud and not es_izq:
                artist.set_visible(False)

    # Colorbar a la derecha
    cbar = fig.colorbar(cf, ax=axes.tolist(), orientation='vertical', 
                        shrink=0.8, pad=0.03, aspect=30)
    cbar.set_label('LIGHTDENS', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    ruta_guardado = generar_ruta_imagen(carpeta="../images", prefijo="comparacion_directa_unet", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Gráfico guardado exitosamente en: {ruta_guardado}")
    
    plt.show()

    print("-" * 50)
    print(f"Estadísticas del Timestep {idx}")
    print(f"WRF-ELEC (Real) — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"{nombre_modelo:<15} — min: {pred_real.min():.2f}, max: {pred_real.max():.2f}, mean: {pred_real.mean():.4f}")

In [ ]:
modelo_bob = BobModel().to(device)
modelo_CNN = BaselineCNN(in_channels=6).to(device)
modelo_unet_WMSE = UNet_WRF(in_channels=6, n_niveles=2, n_filtros_base=32).to(device)
modelo_unet_FACL = UNet_WRF(in_channels=6, n_niveles=2, n_filtros_base=32, usa_sigmoide=True).to(device)
modelo_unet_Comb_V2 = UNet_WRF(in_channels=6, n_niveles=3, n_filtros_base=64, dropout_ps=None, kernel_sizes=[5, 3, 3], usa_sigmoide=True).to(device)

In [ ]:
archivos = sorted(glob.glob('../modelos/*bob.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo_bob.load_state_dict(checkpoint['best_model_state_dict'])
modelo_bob.eval()
resumen_entrenamiento(checkpoint, "bob")

In [ ]:
archivos = sorted(glob.glob('../modelos/*baseline.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo_CNN.load_state_dict(checkpoint['best_model_state_dict'])
modelo_CNN.eval()
resumen_entrenamiento(checkpoint, "CNN")

In [ ]:
archivos = sorted(glob.glob('../modelos/*unet.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo_unet_WMSE.load_state_dict(adaptar_pesos_unet(checkpoint['best_model_state_dict']))
modelo_unet_WMSE.eval()
resumen_entrenamiento(checkpoint, "unet WMSE")

In [ ]:
archivos = sorted(glob.glob('../modelos/*unet_FACL.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo_unet_FACL.load_state_dict(adaptar_pesos_unet(checkpoint['best_model_state_dict']))
modelo_unet_FACL.eval()
resumen_entrenamiento(checkpoint, "unet FACL")

In [ ]:
archivos = sorted(glob.glob('../modelos/*U-Net Comb V2.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo_unet_Comb_V2.load_state_dict(checkpoint['best_model_state_dict'])
modelo_unet_Comb_V2.eval()
resumen_entrenamiento(checkpoint, "U-Net Comb V2")

In [ ]:
modelos = {
    'CNN': modelo_CNN,
    'U-Net WMSE': modelo_unet_WMSE,
    'U-Net Comb V2': modelo_unet_Comb_V2
}

# 1. Ejecutar la búsqueda
mejores_casos = buscar_mejores_ejemplos(
    diccionario_modelos=modelos, 
    dataset=val_dataset, 
    modelos_facl=['U-Net Comb V2'],
    umbral_actividad=1.0  # Asegura que haya algo rojo/amarillo en el mapa real
)

# 2. Tomar el mejor índice (el Puesto 1)
mejor_idx = mejores_casos[0]['idx']

# 3. Graficarlo directamente con tu función
graficar_comparacion_modelos(
    diccionario_modelos=modelos, 
    dataset=val_dataset, 
    idx=mejor_idx,
    modelos_facl=['U-Net Comb V2']
)

In [ ]:
graficar_reconstruccion_individual(
    modelo=modelo_unet_Comb_V2, 
    nombre_modelo='U-Net Comb V2', 
    dataset=val_dataset, 
    idx=236, 
    es_facl=True
)

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo_bob, val_dataset, data_range=data_range_val)

graficar_mse(df_metricas,  nombre_modelo="Bob")

graficar_rmse(df_metricas, nombre_modelo="Bob")

graficar_mae(df_metricas,  nombre_modelo="Bob")

graficar_ssim(df_metricas, nombre_modelo="Bob")

r2 = calcular_r2_global(modelo_bob, val_dataset, nombre_modelo="Bob", ignorar_borde=0)

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo_CNN, val_dataset, data_range=data_range_val)

graficar_mse(df_metricas,  nombre_modelo='CNN')

graficar_rmse(df_metricas, nombre_modelo='CNN')

graficar_mae(df_metricas,  nombre_modelo='CNN')

graficar_ssim(df_metricas, nombre_modelo='CNN')

r2 = calcular_r2_global(modelo_CNN, val_dataset, nombre_modelo='CNN', ignorar_borde=0)

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo_unet_WMSE, val_dataset, data_range=data_range_val)

graficar_mse(df_metricas,  nombre_modelo='U-Net WMSE')

graficar_rmse(df_metricas, nombre_modelo='U-Net WMSE')

graficar_mae(df_metricas,  nombre_modelo='U-Net WMSE')

graficar_ssim(df_metricas, nombre_modelo='U-Net WMSE')

r2 = calcular_r2_global(modelo_unet_WMSE, val_dataset, nombre_modelo='U-Net WMSE', ignorar_borde=0)

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo_unet_FACL, val_dataset, data_range=data_range_val, es_facl=True)

graficar_mse(df_metricas,  nombre_modelo='U-Net FACL')

graficar_rmse(df_metricas, nombre_modelo='U-Net FACL')

graficar_mae(df_metricas,  nombre_modelo='U-Net FACL')

graficar_ssim(df_metricas, nombre_modelo='U-Net FACL')

r2 = calcular_r2_global(modelo_unet_FACL, val_dataset, nombre_modelo='U-Net FACL', ignorar_borde=0, es_facl=True)

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo_unet_Comb_V2, val_dataset, data_range=data_range_val, es_facl=True)

graficar_mse(df_metricas,  nombre_modelo='U-Net Comb V2')

graficar_rmse(df_metricas, nombre_modelo='U-Net Comb V2')

graficar_mae(df_metricas,  nombre_modelo='U-Net Comb V2')

graficar_ssim(df_metricas, nombre_modelo='U-Net Comb V2')

r2 = calcular_r2_global(modelo_unet_Comb_V2, val_dataset, nombre_modelo='U-Net Comb V2', ignorar_borde=0, es_facl=True)

In [ ]:
import sys
!{sys.executable} -m pip install tensorflow

In [ ]:
import os
# Ocultamos la GPU a nivel sistema ANTES de importar TensorFlow
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, ZeroPadding2D, Cropping2D
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Definir la U-Net original (requiere 299x299x3)
# ==========================================
def build_unet(input_shape=(299, 299, 3)):
    inputs = Input(input_shape)
    
    padded_inputs = ZeroPadding2D(padding=((2, 3), (2, 3)))(inputs)

    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(padded_inputs)
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(c3)

    u4 = UpSampling2D((2, 2))(c3)
    u4 = concatenate([u4, c2])
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(c4)

    u5 = UpSampling2D((2, 2))(c4)
    u5 = concatenate([u5, c1])
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(c5)

    outputs_padded = Conv2D(1, (1, 1), activation='linear')(c5)
    outputs = Cropping2D(cropping=((2, 3), (2, 3)))(outputs_padded)

    return Model(inputs=[inputs], outputs=[outputs])

def weighted_mse_log(y_true, y_pred):
    pesos = tf.where(y_true > 0.0, 50.0, 1.0)
    error_cuadrado = K.square(y_true - y_pred)
    return K.mean(error_cuadrado * pesos)

unet_model_viejo = build_unet()
unet_model_viejo.compile(optimizer='adam', loss=weighted_mse_log, metrics=['mae'])

# ==========================================
# 2. Extraer datos del dataset de PyTorch y adaptarlos a Keras
# ==========================================
def preparar_lote(dataset, num_samples=30):
    X_list, y_list = [], []
    max_idx = min(num_samples, len(dataset))
    
    for i in range(max_idx):
        x, y = dataset[i]
        x_np = x.numpy().transpose(1, 2, 0)
        y_np = y.numpy().transpose(1, 2, 0)

        # Padding espacial de 297x297 a 299x299 (agregamos 1 píxel a cada borde)
        x_pad = np.pad(x_np, ((1, 1), (1, 1), (0, 0)), mode='constant')
        y_pad = np.pad(y_np, ((1, 1), (1, 1), (0, 0)), mode='constant')

        # Recortar canales (solo nos quedamos con los primeros 3)
        X_list.append(x_pad[:, :, :3])
        y_list.append(y_pad)

    return np.array(X_list), np.array(y_list)

print("Extrayendo y adaptando datos a la CPU...")
# NOTA: Asegurate de tener train_dataset y val_dataset cargados en memoria
# (ejecutá las celdas de PyTorch donde los definís antes de correr esta celda)
X_train_k, y_train_k = preparar_lote(train_dataset, num_samples=40)
X_val_k, y_val_k = preparar_lote(val_dataset, num_samples=10)

y_train_log = np.log1p(y_train_k)
y_val_log = np.log1p(y_val_k)

# ==========================================
# 3. Reentrenar el modelo
# ==========================================
print("Reentrenando el modelo original (usando CPU)...")
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = unet_model_viejo.fit(
    X_train_k, y_train_log,
    validation_data=(X_val_k, y_val_log),
    epochs=10,
    batch_size=4,
    callbacks=[early_stop],
    verbose=1
)

# ==========================================
# 4. Graficar (Replicar la imagen)
# ==========================================
idx = 3
muestra_x = np.expand_dims(X_val_k[idx], axis=0)
muestra_y_real = y_val_log[idx]

prediccion_log = unet_model_viejo.predict(muestra_x)
prediccion_real_scale = np.expm1(prediccion_log)
prediccion_grafico = prediccion_real_scale[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(X_val_k[idx, :, :, 0], cmap='Blues', origin='lower')
axes[0].set_title('Entrada: Maxice (Adaptado a 299x299)')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(np.expm1(muestra_y_real[:, :, 0]), cmap='magma', origin='lower')
axes[1].set_title('Real: Densidad de Rayos')
fig.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(prediccion_grafico[:, :, 0], cmap='magma', origin='lower')
axes[2].set_title('Predicción: UNET Viejo')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 4. Graficar y Guardar (Colorbars verticales y estilizadas)
# ==========================================
idx = 3
muestra_x = np.expand_dims(X_val_k[idx], axis=0)
muestra_y_real = y_val_log[idx]

prediccion_log = unet_model_viejo.predict(muestra_x)
prediccion_real_scale = np.expm1(prediccion_log)
prediccion_grafico = prediccion_real_scale[0]

# Ampliamos la figura a lo ancho para dar espacio
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Entrada Maxice
im0 = axes[0].imshow(X_val_k[idx, :, :, 0], cmap='Greys', origin='lower')
axes[0].set_title('Entrada: Maxice')
axes[0].set_xlim(210, 299)
axes[0].set_ylim(0, 50)
# shrink achica el largo y fraction achica el grosor
fig.colorbar(im0, ax=axes[0], shrink=0.5, fraction=0.046, pad=0.04)

# Plot 2: Real Densidad de Rayos
im1 = axes[1].imshow(np.expm1(muestra_y_real[:, :, 0]), cmap='gray', origin='lower')
axes[1].set_title('Real: Densidad de Rayos')
axes[1].set_xlim(210, 299)
axes[1].set_ylim(0, 50)
fig.colorbar(im1, ax=axes[1], shrink=0.5, fraction=0.046, pad=0.04)

# Plot 3: Predicción Baseline
im2 = axes[2].imshow(prediccion_grafico[:, :, 0], cmap='gray', origin='lower')
axes[2].set_title('Predicción: Baseline')
axes[2].set_xlim(210, 299)
axes[2].set_ylim(0, 50)
fig.colorbar(im2, ax=axes[2], shrink=0.5, fraction=0.046, pad=0.04)

plt.tight_layout()

# Guardamos la imagen en la carpeta figs con alta resolución (300 DPI)
plt.savefig('../figs/figura_3_baseline.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ==========================================
# 4. CONSTRUCCIÓN DEL MODELO U-NET (Adaptado para 299x299)
# ==========================================
print("\n4. Construyendo arquitectura U-Net...")

def build_unet(input_shape=(299, 299, 3)):
    inputs = Input(input_shape)

    # --- Ajuste interno de dimensiones ---
    # Rellenamos con ceros para llevar la imagen de 299x299 a 304x304 (divisible por 16)
    padded_inputs = ZeroPadding2D(padding=((2, 3), (2, 3)))(inputs)

    # --- Bajada (Encoder) ---
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(padded_inputs)
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1) # Baja a 152x152

    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2) # Baja a 76x76
    
    # --- Cuello de botella (Bottleneck) ---
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(c3)

    # --- Subida (Decoder) ---
    u4 = UpSampling2D((2, 2))(c3) # Sube a 152x152
    u4 = concatenate([u4, c2]) 
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(c4)

    u5 = UpSampling2D((2, 2))(c4) # Sube a 304x304
    u5 = concatenate([u5, c1]) 
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(c5)

    # --- Capa de Salida y Reajuste ---
    outputs_padded = Conv2D(1, (1, 1), activation='linear')(c5)
    outputs = Cropping2D(cropping=((2, 3), (2, 3)))(outputs_padded)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

unet_model = build_unet()

# Compilación
unet_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ==========================================
# 5. ENTRENAMIENTO DEL MODELO
# ==========================================
print("\n5. Iniciando Entrenamiento...")

# Asignamos las variables de entrenamiento extraídas previamente
X_train_norm = X_train_k
X_val_norm = X_val_k

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = unet_model.fit(
    X_train_norm, y_train_log,
    validation_data=(X_val_norm, y_val_log),
    epochs=50, 
    batch_size=4, 
    callbacks=[early_stop],
    verbose=1
)

print("\n¡Entrenamiento finalizado exitosamente!")

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. Obtención e Inferencia de Datos (idx = 3)
# ==========================================
idx = 3

# --- Panel (a): Real ---
y_real_plot = np.expm1(y_val_log[idx][:, :, 0])

# --- Panel (b): Predicción Baseline (Keras) ---
muestra_x_keras = np.expand_dims(X_val_k[idx], axis=0)
pred_baseline_log = unet_model_viejo.predict(muestra_x_keras)
pred_baseline_plot = np.expm1(pred_baseline_log[0, :, :, 0])

# --- Panel (c): Predicción U-Net (Keras) ---
muestra_x_unet = np.expand_dims(X_val_k[idx], axis=0)
pred_unet_log = unet_model.predict(muestra_x_unet)
pred_unet_plot = np.expm1(pred_unet_log[0, :, :, 0])

# ==========================================
# 2. Graficado (Dominio completo 299x299)
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel (a) Real
im0 = axes[0].imshow(y_real_plot, cmap='gray', origin='lower')
axes[0].set_title('Real: Densidad de Rayos', pad=8)
axes[0].text(0.5, 1.10, '(a) Real', transform=axes[0].transAxes, ha='center', fontsize=12)
fig.colorbar(im0, ax=axes[0], shrink=0.75, fraction=0.046, pad=0.04)

# Panel (b) Baseline
im1 = axes[1].imshow(pred_baseline_plot, cmap='gray', origin='lower')
axes[1].set_title('Predicción: Baseline', pad=8)
axes[1].text(0.5, 1.10, '(b) Baseline', transform=axes[1].transAxes, ha='center', fontsize=12)
fig.colorbar(im1, ax=axes[1], shrink=0.75, fraction=0.046, pad=0.04)

# Panel (c) U-Net
im2 = axes[2].imshow(pred_unet_plot, cmap='gray', origin='lower')
axes[2].set_title('Predicción: UNET', pad=8)
axes[2].text(0.5, 1.10, '(c) U-Net', transform=axes[2].transAxes, ha='center', fontsize=12)
fig.colorbar(im2, ax=axes[2], shrink=0.75, fraction=0.046, pad=0.04)

plt.tight_layout()

# ==========================================
# 3. Guardar en Alta Resolución
# ==========================================
# Creamos la carpeta automáticamente si no existe
os.makedirs('../figs', exist_ok=True)

plt.savefig('../figs/figura_4_comparacion_completa.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
"""
Genera 6 mapas (CTT, QICE+QHAIL+QGRAUP+QSNOW, LPI, MDBZ, QCLOUD+QRAIN, WMAX)
para un timestep elegido al azar entre los archivos WRF disponibles en las
dos carpetas de salidas (2018 y 2019), y los guarda en el directorio actual.

Basado en las funciones de extracción de Datasets.ipynb (transform_Q, get_LPI,
calcular_lpi, calcular_intice_tmp) y en el patrón de ploteo con cartopy usado
en ese mismo notebook.
"""

import os
import glob
import random
from copy import deepcopy

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as crs
import cartopy.feature as cfeature
from netCDF4 import Dataset
from wrf import getvar, to_np, get_cartopy, latlon_coords
from metpy import constants

# ==========================================
# RUTAS A LAS DOS CARPETAS DE SALIDAS WRF
# ==========================================
DIR_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
DIR_2019 = "../salidasWRF2019"
PATRON_ARCHIVO = "wrfout_d01_*"

# ==========================================
# PARCHE para poder usar use_varint/use_liqskin con mdbz (igual que en Datasets.ipynb)
# ==========================================
from wrf.routines import _VALID_KARGS
_VALID_KARGS["maxdbz"] = ["use_varint", "use_liqskin"]


# ==========================================
# FUNCIONES DE EXTRACCIÓN (copiadas de Datasets.ipynb)
# ==========================================
def transform_Q(ds_Q, var):
    for v in var:
        ds_Q[v] = ds_Q[v].where(ds_Q[v] >= 0, 0) * 1e-3
    return ds_Q


def get_LPI(ds):
    ds_LPI = deepcopy(ds[['W', 'T', 'GEOPT', 'QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD']])
    ds_LPI = transform_Q(ds_LPI, ['QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD'])
    ds_LPI['GEOPT'] = ds_LPI['GEOPT'] / constants.g

    del_Z = ds_LPI['GEOPT'][1:, :, :].values - ds_LPI['GEOPT'][:-1, :, :].values
    ds_LPI = ds_LPI.isel(bottom_top=slice(0, -1))

    flag_T = (ds_LPI['T'] > (-20 + 273.15)) * (ds_LPI['T'] < 273.15)
    ds_LPI = ds_LPI.where(flag_T)

    Q_l = ds_LPI['QCLOUD'] + ds_LPI['QRAIN']

    den_i = ds_LPI['QICE'] + ds_LPI['QGRAUP']
    num_i = np.sqrt(ds_LPI['QICE'] * ds_LPI['QGRAUP'])
    flag_i = (ds_LPI['QICE'] != 0) + (ds_LPI['QGRAUP'] != 0)
    den_i = den_i.where(flag_i, 10000)
    num_i = num_i.where(flag_i, 0)

    den_s = ds_LPI['QSNOW'] + ds_LPI['QGRAUP']
    num_s = np.sqrt(ds_LPI['QSNOW'] * ds_LPI['QGRAUP'])
    flag_s = (ds_LPI['QSNOW'] != 0) + (ds_LPI['QGRAUP'] != 0)
    den_s = den_s.where(flag_s, 10000)
    num_s = num_s.where(flag_s, 0)

    q_isg = ds_LPI['QGRAUP'] * (num_i / den_i + num_s / den_s)

    num_l = np.sqrt(Q_l * q_isg)
    den_l = Q_l + q_isg
    flag_l = (Q_l != 0) + (q_isg != 0)
    den_l = den_l.where(flag_l, 10000)
    num_l = num_l.where(flag_l, 0)

    lpi_tmp = ((del_Z * (2 * (num_l / den_l)) * ds_LPI['W'] ** 2)).sum(dim='bottom_top')
    lpi = lpi_tmp / (del_Z * flag_T).sum(dim='bottom_top')
    return lpi


def calcular_lpi(ds):
    """Adaptador: recibe un Dataset de netCDF4, devuelve LPI como numpy array."""
    variables = ['tk', 'wa', 'geopt', 'QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD']
    ds_list = [getvar(ds, var) for var in variables]
    ds_xr = xr.combine_by_coords(ds_list, compat='override')
    ds_xr = ds_xr.rename({'temp': 'T', 'wa': 'W', 'geopt': 'GEOPT'})
    lpi = get_LPI(ds_xr)
    return to_np(lpi)


def calcular_intice_tmp(ds):
    """QICE+QHAIL+QGRAUP+QSNOW en la banda de temperatura de electrificación (-10 a -40 C)."""
    variables = ['temp', 'QGRAUP', 'QHAIL', 'QICE', 'QSNOW']
    ds_list = [getvar(ds, v, units="degC") if v == 'temp' else getvar(ds, v) for v in variables]
    ds_xr = xr.combine_by_coords(ds_list, compat='override')

    qsum = ds_xr['QGRAUP'] + ds_xr['QHAIL'] + ds_xr['QICE'] + ds_xr['QSNOW']
    qsum = qsum.fillna(0)

    mask = (ds_xr['temp'] <= -10) & (ds_xr['temp'] >= -40)
    qmask = qsum.where(mask, 0)
    return np.array(qmask.sum(dim="bottom_top", skipna=True))


# ==========================================
# CONFIGURACIÓN DE LAS 6 VARIABLES A GRAFICAR (en inglés)
# ==========================================
VAR_CONFIGS = [
    {"nombre": "CTT", "unidades": "°C", "cmap": "RdBu_r", "keys": ["ctt"]},
    {"nombre": "QICE+QHAIL+QGRAUP+QSNOW", "unidades": "kg/kg", "cmap": "Blues",
     "func": calcular_intice_tmp},
    {"nombre": "LPI", "unidades": "J/kg", "cmap": "YlOrRd", "func": calcular_lpi},
    {"nombre": "MDBZ", "unidades": "dBZ", "cmap": "jet", "keys": ["mdbz"], "clip_min": 0,
     "getvar_kwargs": {"use_varint": True, "use_liqskin": True}},
    {"nombre": "QCLOUD+QRAIN", "unidades": "kg/kg", "cmap": "Blues", "keys": ["QCLOUD", "QRAIN"]},
    {"nombre": "WMAX", "unidades": "m/s", "cmap": "RdBu_r", "keys": ["wa"]},
]


def extraer_variable(ds, var_config):
    """Extrae una variable (o suma de variables, o función personalizada) de un Dataset abierto."""
    if "func" in var_config:
        return np.array(var_config["func"](ds))

    resultado = None
    kwargs = var_config.get("getvar_kwargs", {})
    for key in var_config["keys"]:
        datos = to_np(getvar(ds, key, **kwargs))
        if datos.ndim == 3:
            datos = datos.max(axis=0)
        resultado = datos if resultado is None else resultado + datos

    if "clip_min" in var_config:
        resultado = np.maximum(resultado, var_config["clip_min"])
    return resultado


def elegir_archivo_random(seed=None):
    """Elige un archivo al azar entre las dos carpetas de salidas WRF."""
    if seed is not None:
        random.seed(seed)

    archivos = sorted(
        glob.glob(os.path.join(DIR_2018, PATRON_ARCHIVO)) +
        glob.glob(os.path.join(DIR_2019, PATRON_ARCHIVO))
    )
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos con el patrón '{PATRON_ARCHIVO}' en:\n"
            f"  {DIR_2018}\n  {DIR_2019}\n"
            "Revisá las rutas y el patrón de nombre al principio del script."
        )
    return random.choice(archivos)


def main():
    filepath = elegir_archivo_random()
    print(f"Archivo elegido al azar: {filepath}")

    ds = Dataset(filepath)

    # Coordenadas y proyección (usamos LIGHTDENS solo para los metadatos, como en Datasets.ipynb)
    ref = getvar(ds, "LIGHTDENS")
    cart_proj = get_cartopy(ref)
    lats, lons = latlon_coords(ref)
    tiempo = str(ref.Time.values)[:16]

    nombre_archivo = os.path.basename(filepath).replace(":", "-")

    for var_config in VAR_CONFIGS:
        datos = extraer_variable(ds, var_config)

        fig = plt.figure(figsize=(9, 7))
        ax = plt.axes(projection=cart_proj)
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor="black")

        cf = ax.contourf(
            to_np(lons), to_np(lats), datos,
            transform=crs.PlateCarree(),
            cmap=var_config["cmap"], levels=20,
        )
        fig.colorbar(cf, ax=ax, label=var_config["unidades"], fraction=0.046, pad=0.04)

        nombre_salida = f"{var_config['nombre'].replace('+', '_')}.png"
        plt.tight_layout()
        plt.savefig(nombre_salida, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Guardado: {nombre_salida}")

    ds.close()
    print("\nListo — 6 imágenes generadas en el directorio actual.")


if __name__ == "__main__":
    main()

In [ ]:
import sys
import os
import glob
import random
import torch

# Agregamos la ruta absoluta de 'Trabajo_Final' a las rutas del sistema
ruta_base = '/serafin/gcamilo/Trabajo_Final'
if ruta_base not in sys.path:
    sys.path.append(ruta_base)

from modulos.unet_diagram import draw_unet

modelo = UNet_WRF(
    in_channels=3,
    n_niveles=3,
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3],
    dropout_ps=[0.0, 0.0, 0.0]
).to(device)

archivos = sorted(glob.glob("../modelos/*Ext U-Net Comb V2.1*.pth"))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo.load_state_dict(checkpoint['best_model_state_dict'])
modelo.eval()

# --- Seleccionar índice aleatorio e imprimirlo ---
idx = random.randint(0, len(val_dataset) - 1)
print(f"🎲 Muestra seleccionada aleatoriamente: idx = {idx}")

X, y = val_dataset[idx]

with torch.no_grad():
    pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

y_pred = val_dataset.desnormalizar_y(pred_log).numpy()
X_muestra = X.numpy()

# bbox a partir de las coordenadas reales que ya tenés cargadas (celda 0)
lon_min, lon_max = float(LONS.min()), float(LONS.max())
lat_min, lat_max = float(LATS.min()), float(LATS.max())

draw_unet(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64, 
    kernel_sizes=[5, 3, 3],
    X_sample=X_muestra,
    y_sample=None,              # Sin target
    y_pred_sample=y_pred,        # Solo la predicción
    var_names=["mdbz", "hielo", "ctt"],
    var_cmaps=["jet", "Blues", "RdBu_r"],
    target_cmap="hot_r",
    lon_min=lon_min, lon_max=lon_max, lat_min=lat_min, lat_max=lat_max,
    out_path="unet_arquitectura.png",
)

In [ ]:
def agrupar_indices_por_tormenta(dataset):
    """
    Agrupa los índices de `dataset` (TormentaDataset) en tormentas: tramos de
    archivos consecutivos en el tiempo (mismo paso que el resto del split).
    Como el dataset de val solo tiene los samples de las tormentas de
    validación (sin los timesteps de las tormentas de train intercaladas),
    un salto de tiempo entre dos archivos consecutivos marca el límite entre
    una tormenta y la siguiente. No depende de una lista de fechas mantenida
    a mano: si val_tormentas cambia, esto se ajusta solo.

    Devuelve dict id_tormenta -> lista de índices (ordenados), donde
    id_tormenta es la fecha/hora del primer archivo ('YYYY-MM-DD_HHMM').
    """
    n = len(dataset)
    if n == 0:
        return {}
    ts = []
    for idx in range(n):
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})', dataset.files[idx].name)
        fecha, h, mi, s = m.group(1), m.group(2), m.group(3), m.group(4)
        ts.append((idx, datetime.strptime(f"{fecha} {h}:{mi}:{s}", "%Y-%m-%d %H:%M:%S")))
    ts.sort(key=lambda x: x[1])
    if n == 1:
        return {f"{ts[0][1]:%Y-%m-%d_%H%M}": [ts[0][0]]}

    paso = min(b[1] - a[1] for a, b in zip(ts[:-1], ts[1:]))
    grupos = {}
    grupo_actual = [ts[0]]
    for a, b in zip(ts[:-1], ts[1:]):
        if b[1] - a[1] == paso:
            grupo_actual.append(b)
        else:
            clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
            grupos[clave] = [i for i, _ in grupo_actual]
            grupo_actual = [b]
    clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
    grupos[clave] = [i for i, _ in grupo_actual]
    return grupos

def animar_reconstruccion_dia(modelo, dataset, fecha, nombre_modelo='Model', fps=5, vmax_percentil=99, umbral_ruido=0.5, aviso_gap_min=10):
    """
    Arma una única animación con todas las tormentas de `dataset` que arrancan
    el día `fecha` ('YYYY-MM-DD'), concatenadas en orden temporal.

    Como `dataset` solo tiene los timesteps que sobrevivieron al filtrado por
    tormenta (no el día completo de 720 timesteps), puede haber saltos de
    tiempo entre frames consecutivos cuando hay más de una tormenta ese día.
    Esos saltos se listan antes de renderizar, no se ocultan.
    """
    modelo.eval()

    grupos = agrupar_indices_por_tormenta(dataset)
    indices = sorted(i for tid, idxs in grupos.items() if tid.startswith(fecha) for i in idxs)
    if not indices:
        raise ValueError(f"No hay samples de tormentas que arranquen el {fecha} en este dataset.")

    print(f"=== Generating animation for {fecha} ({len(indices)} timesteps, {sum(1 for tid in grupos if tid.startswith(fecha))} tormenta(s)) ===")

    # --- Timestamps reales de cada frame, para detectar saltos y titular cada cuadro ---
    tiempos = []
    for idx in indices:
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})', dataset.files[idx].name)
        fecha_f, h, mi, s = m.group(1), m.group(2), m.group(3), m.group(4)
        tiempos.append(datetime.strptime(f"{fecha_f} {h}:{mi}:{s}", "%Y-%m-%d %H:%M:%S"))

    paso = min((b - a for a, b in zip(tiempos[:-1], tiempos[1:])), default=timedelta(0))
    saltos = [(i, tiempos[i], tiempos[i + 1]) for i in range(len(tiempos) - 1)
              if tiempos[i + 1] - tiempos[i] > max(paso, timedelta(minutes=aviso_gap_min))]
    if saltos:
        print(f"⚠ {len(saltos)} salto(s) de tiempo entre frames (timesteps inactivos no incluidos en el dataset):")
        for i, t0, t1 in saltos:
            print(f"   frame {i} → {i+1}: {t0:%H:%M} → {t1:%H:%M} ({t1 - t0})")

    # --- Cachear datos y predicciones ---
    predicciones_cache, reales_cache = [], []
    for idx in tqdm(indices, desc=f"Calculating range — {fecha}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()
        y_pred[y_pred < umbral_ruido] = np.nan
        y_real[y_real < umbral_ruido] = np.nan
        reales_cache.append(y_real)
        predicciones_cache.append(y_pred)

    n_frames = len(indices)
    valores_activos = np.concatenate([r[~np.isnan(r)] for r in reales_cache])
    vmax_crudo = np.percentile(valores_activos, vmax_percentil) if len(valores_activos) > 0 else 10.0

    paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
    if paso_base >= 10:
        x = int(np.ceil(paso_base / 10.0)) * 10
    elif paso_base > 5:
        x = int(np.ceil(paso_base / 5.0)) * 5
    else:
        x = paso_base
    vmax = int(np.ceil(vmax_crudo / x)) * x
    niveles = np.arange(0, vmax + x, x)
    print(f"LIGHTDENS Range ({fecha}): 0 - {vmax} (Step: {x})")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
    fig.suptitle(f'{nombre_modelo} — {fecha}', fontsize=15, y=0.98)

    ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
    ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

    def dibujar_fondo(ax):
        ax.set_facecolor('white')
        ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 8, 'color': 'gray'}
        gl.ylabel_style = {'size': 8, 'color': 'gray'}

    for ax in axes:
        dibujar_fondo(ax)

    pbar = tqdm(total=n_frames, desc="Rendering Animation")

    def update(frame_idx):
        for ax in axes:
            ax.cla()
            dibujar_fondo(ax)

        t = tiempos[frame_idx]
        marca = " ⚠" if any(frame_idx == i + 1 for i, _, _ in saltos) else ""

        cf0 = axes[0].contourf(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap=CMAP_LIGHTDENS, levels=niveles, vmin=0, vmax=vmax, extend='max')
        axes[0].set_title(f'Ground Truth: LIGHTDENS — {t:%H:%M}{marca}')

        cf1 = axes[1].contourf(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap=CMAP_LIGHTDENS, levels=niveles, vmin=0, vmax=vmax, extend='max')
        axes[1].set_title(f'Prediction: {nombre_modelo}')

        pbar.update(1)
        return cf0, cf1

    cf_inicial_real, cf_inicial_pred = update(0)

    fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=niveles)
    fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=niveles)

    plt.tight_layout()

    ani = animation.FuncAnimation(fig, update, frames=range(n_frames), interval=200)

    carpeta_destino = "../videos&gifs"
    os.makedirs(carpeta_destino, exist_ok=True)
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_final = os.path.join(carpeta_destino, f"reconstruction_{nombre_limpio}_dia_{fecha}.gif")

    ani.save(ruta_final, writer="pillow", fps=fps)
    pbar.close()
    plt.close(fig)

    print(f"Animation saved to {ruta_final}")
    return ruta_final

In [ ]:
# -*- coding: utf-8 -*-
"""
Búsqueda de un instante "lindo" para las figuras de la monografía
(Fig. 1.1, Fig. 3.1 y Fig. 3.4), a partir del dataset temporal.

Qué hace
--------
1. ranking(): recorre el dataset (por defecto, validación) y mide en
   cada muestra cuántos núcleos de descargas hay, qué tan intensos
   son, qué tan altas/frías son MDBZ/CTT y cuánto cambia HIELO
   respecto del instante anterior. Devuelve una tabla ordenada por
   un puntaje (los pesos están en PESOS, se pueden tocar).
2. hoja_de_contacto(): dibuja los mejores candidatos en una sola
   imagen (MDBZ, HIELO, CTT, ΔHIELO y LIGHTDENS) para elegir a ojo.
3. exportar_instante(): guarda en un .npz el instante elegido (los
   9 canales, LIGHTDENS real y el contenido crudo del .npz del
   instante) para pasárselo a Claude y dibujar los mapas.

Uso en JupyterHub (después de crear val_dataset y NOMBRES_CANALES
como en la notebook de entrenamiento)
------------------------------------------------------------
    import sys; sys.path.append('../figuras')   # o copiar el archivo
    from buscar_instante import (ranking, hoja_de_contacto,
                                 exportar_instante)

    tabla = ranking(val_dataset, NOMBRES_CANALES)
    print(tabla.head(15))
    hoja_de_contacto(val_dataset, tabla, NOMBRES_CANALES, k=8)
    exportar_instante(val_dataset, int(tabla.idx.iloc[0]),
                      NOMBRES_CANALES, 'instante_elegido.npz')

Notas
-----
* Supone que dataset[idx] devuelve (X, y) con X de 9 canales
  normalizados y que dataset.desnormalizar_y y dataset.files existen
  (como en la notebook de entrenamiento).
* Para la Fig. 3.1 también hacen falta AGUA, LPI y WMAX en ese
  mismo instante: calculalos desde el wrfout con tus funciones de
  Datasets.ipynb y pasalos en `extras` de exportar_instante.
* Idealmente, el instante sale de una de las 4 tormentas de
  validación (14/12/2018, 29/01/2019, 09/02/2019, 04/03/2019), y
  se usa el mismo en las tres figuras.
"""
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

# Pesos del puntaje (se aplican a los rangos percentiles, 0 a 1).
PESOS = {
    "n_nucleos": 3.0,    # varios núcleos separados
    "y_max": 1.0,        # algún núcleo intenso
    "mdbz_max": 1.0,     # reflectividad alta
    "ctt_min": 1.0,      # topes fríos
    "d1_cambio": 1.5,    # HIELO cambió visiblemente en t-1 -> t
    "d1_balance": 1.0,   # crecimiento y decaimiento a la vez
}


def _np(t):
    return t.detach().cpu().numpy() if hasattr(t, "detach") \
        else np.asarray(t)


def _y_real(dataset, y):
    """LIGHTDENS en unidades reales (recortado a 150), como en
    graficar_reconstruccion de la notebook."""
    yy = dataset.desnormalizar_y(y.squeeze())
    return np.round(_np(yy))


def _metricas_muestra(dataset, idx, nombres, min_px, margen):
    X, y = dataset[idx]
    X = _np(X)
    yr = _y_real(dataset, y)
    act = yr > 0
    lab, n = ndi.label(act, structure=np.ones((3, 3), int))
    if n:
        tam = ndi.sum(act, lab, index=np.arange(1, n + 1))
        n_nuc = int((tam >= min_px).sum())
    else:
        n_nuc = 0
    toca_borde = bool(act[:margen].any() or act[-margen:].any()
                      or act[:, :margen].any()
                      or act[:, -margen:].any())
    d = X[nombres.index("HIELO_d1")]
    d = d - np.median(d)     # el 0 de la diferencia, en la escala
    pos, neg = float(d.max()), float(-d.min())   # normalizada
    return {
        "idx": idx,
        "archivo": dataset.files[idx].name,
        "n_activos": int(act.sum()),
        "n_nucleos": n_nuc,
        "y_max": float(yr.max()),
        "mdbz_max": float(X[nombres.index("MDBZ_t")].max()),
        "ctt_min": float(X[nombres.index("CTT_t")].min()),
        "d1_cambio": max(pos, neg),
        "d1_balance": min(pos, neg) / max(pos, neg, 1e-9),
        "toca_borde": toca_borde,
    }


def ranking(dataset, nombres, min_px=3, margen=8, paso=1,
            solo_fechas=None, min_nucleos=2, y_max_tope=140,
            pesos=PESOS):
    """Tabla de candidatos ordenada por puntaje.

    min_px      : tamaño mínimo (en píxeles) para contar un núcleo.
    margen      : se descartan las muestras con descargas a menos de
                  `margen` píxeles del borde.
    paso        : 1 = todas las muestras; 2 = una de cada dos, etc.
    solo_fechas : tupla de prefijos 'AAAA-MM-DD' para filtrar por el
                  nombre del archivo (por ej. ('2018-12-14',)).
    """
    filas = []
    for idx in range(0, len(dataset), paso):
        nombre = dataset.files[idx].name
        if solo_fechas and not any(f in nombre for f in solo_fechas):
            continue
        filas.append(_metricas_muestra(dataset, idx, nombres,
                                       min_px, margen))
    t = pd.DataFrame(filas)
    if t.empty:
        raise ValueError("No hay muestras que cumplan el filtro.")
    # Rangos percentiles entre las muestras con actividad.
    pct = pd.DataFrame({
        "n_nucleos": t.n_nucleos.clip(upper=6).rank(pct=True),
        "y_max": t.y_max.clip(upper=y_max_tope).rank(pct=True),
        "mdbz_max": t.mdbz_max.rank(pct=True),
        "ctt_min": (-t.ctt_min).rank(pct=True),
        "d1_cambio": t.d1_cambio.rank(pct=True),
        "d1_balance": t.d1_balance.rank(pct=True),
    })
    t["score"] = sum(w * pct[k] for k, w in pesos.items()) \
        / sum(pesos.values())
    ok = (t.n_nucleos >= min_nucleos) & (~t.toca_borde) \
        & (t.y_max <= y_max_tope)
    t = t[ok].sort_values("score", ascending=False)
    return t.reset_index(drop=True).round(3)


def hoja_de_contacto(dataset, tabla, nombres, k=8,
                     ruta="candidatos.png"):
    """Imagen con los k mejores candidatos: MDBZ, HIELO, CTT,
    ΔHIELO (t - (t-1)) y LIGHTDENS real."""
    import matplotlib
    import matplotlib.pyplot as plt
    k = min(k, len(tabla))
    cols = [("MDBZ_t", "viridis"), ("HIELO_t", "viridis"),
            ("CTT_t", "viridis_r"), ("HIELO_d1", "RdBu_r"),
            ("LIGHTDENS", "magma")]
    fig, axs = plt.subplots(k, len(cols), figsize=(11, 2.1 * k),
                            squeeze=False)
    for i in range(k):
        idx = int(tabla.idx.iloc[i])
        X, y = dataset[idx]
        X = _np(X)
        yr = _y_real(dataset, y)
        for j, (nom, cmap) in enumerate(cols):
            ax = axs[i, j]
            m = yr if nom == "LIGHTDENS" else X[nombres.index(nom)]
            kw = {}
            if nom == "HIELO_d1":
                c = np.median(m)
                r = max(abs(m - c).max(), 1e-9)
                kw = dict(vmin=c - r, vmax=c + r)
            ax.imshow(m, origin="lower", cmap=cmap, **kw)
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(nom, fontsize=9)
        axs[i, 0].set_ylabel(
            f"idx {idx}\n{tabla.n_nucleos.iloc[i]} núcleos\n"
            f"máx {tabla.y_max.iloc[i]:.0f}", fontsize=7)
        axs[i, 2].set_xlabel(tabla.archivo.iloc[i][11:-4],
                             fontsize=7)
    fig.tight_layout()
    fig.savefig(ruta, dpi=110)
    plt.close(fig)
    print("Guardado:", ruta)


def exportar_instante(dataset, idx, nombres, ruta, extras=None):
    """Guarda el instante `idx` en un .npz para dibujar las figuras.

    Contenido: X (9 canales normalizados, tal como los ve la red),
    nombres_canales, y_real (LIGHTDENS), archivo, y todo lo que
    haya en el .npz del instante en disco con el prefijo 'crudo_'
    (valores físicos, si el archivo los guarda así). `extras` es un
    dict opcional, por ejemplo {'AGUA': ..., 'LPI': ..., 'WMAX':
    ...}, calculados desde el wrfout de ese instante."""
    X, y = dataset[idx]
    datos = {
        "X": _np(X),
        "nombres_canales": np.array(nombres),
        "y_real": _y_real(dataset, y),
        "archivo": np.array(dataset.files[idx].name),
        "idx": np.array(idx),
    }
    with np.load(dataset.files[idx]) as f:
        print("Contenido del .npz del instante:",
              {k: f[k].shape for k in f.files})
        for k in f.files:
            datos["crudo_" + k] = f[k]
    for k, v in (extras or {}).items():
        datos["extra_" + k] = np.asarray(v)
    np.savez_compressed(ruta, **datos)
    print("Guardado:", ruta, "-", dataset.files[idx].name)


In [ ]:
tabla = ranking(val_dataset, NOMBRES_CANALES)
print(tabla.head(15))
hoja_de_contacto(val_dataset, tabla, NOMBRES_CANALES, k=8)

In [ ]:
exportar_instante(val_dataset, 1924, NOMBRES_CANALES, 'instante_1924.npz')